---
title: "Download: `download` Module as a `pytask` Task"
---

## task_download 

> This module downloads the raw era5 data from the CDS API. It is similar to the original script, refactored for `pytask`.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

We're going to quickly refactor the pipeline to use pytask instead of hydra and snakemake. This will hopefully demonstrate a simpler and more flexible way to manage data pipelines in Python.

To start off, we need to create a function that queries the CDS API with one job. This function will be used to download the data for each query in the range specified in the data catalog in the config file.

Let's take a look at the data catalog we created in the config module:

You can see the queries entry we created in the data catalog. Each query is a namedtuple that contains the parameters for the CDS API query. The `query` namedtuple has the following variable fields (other fields are singletons): `year`, `month`, `geography`, and `variable`.

In [ ]:
queries = data_catalog['queries'].load()
queries[-3:]

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[1], line 1
----> 1 queries = data_catalog['queries'].load()
      2 queries[-3:]

KeyError: 'queries'


KeyError: 'queries'

We can test this query like we did in the original work:

In [ ]:
example_query = queries[0]

create_bounding_box(example_query.geography['shapefile'])

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 1
----> 1 example_query = queries[0]
      3 create_bounding_box(example_query.geography['shapefile'])

NameError: name 'queries' is not defined


NameError: name 'queries' is not defined

In this way, we have a similar approach as Hydra configs, but, using the `pytask` data catalog, we can more easily gather the data for a specific task in structured manner entirely in Python.

In [ ]:
#| eval: false
client = cdsapi.Client()

ex_bounding_box = create_bounding_box(example_query.geography['shapefile'])

request = {
            "product_type": example_query.product_type,
            "variable": example_query.variables, 
            "year": example_query.year,
            "month": example_query.month,
            "day": example_query.day,
            "time": example_query.time,
            "data_format": "netcdf",
            "download_format": "unarchived",
            "area": ex_bounding_box
        }

target = f"{example_query.name()}.nc"

client.retrieve("reanalysis-era5-land", request).download(target)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 5
      1 #| eval: false
      3 client = cdsapi.Client()
----> 5 ex_bounding_box = create_bounding_box(example_query.geography['shapefile'])
      7 request = {
      8             "product_type": example_query.product_type,
      9             "variable": example_query.variables, 
   (...)     16             "area": ex_bounding_box
     17         }
     19 target = f"{example_query.name()}.nc"

NameError: name 'example_query' is not defined


2025-08-15 11:22:18,221 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.


NameError: name 'example_query' is not defined

This works! So now we just need to create a `task_` function that pytask will recognise to parallelise the download of queries over:

### How this works (with some help from GPT):

#### 🧠 How pytask Discovers and Executes Tasks

When you run pytask, it automatically scans your project for Python files named `task_*.py`. In these files, it looks for:
- Functions decorated with `@task`, or
- Functions prefixed with `task_`

These functions are not executed immediately. Instead, `pytask`:
1.	Imports each task_*.py module (just like Python would)
2.	Registers any matching task functions as nodes in a directed acyclic graph (DAG)
3.	Resolves dependencies by analyzing:
    - Input annotations (e.g., `Annotated[x, DependsOn]`)
    - Output declarations (e.g., `return` values or `Product` annotations)
4.	Builds the DAG, where each task function is a node
5.	Executes the tasks, respecting dependency order and skipping up-to-date nodes

So even though the task functions aren’t explicitly “run” in the Python code itself, pytask knows how and when to execute them — based on their position in the DAG.

#### 🔄 How This Differs from Snakemake

In `snakemake`, you’re expected to define a series of explicitly executable rules, often using shell commands or Python scripts. You “stitch together” rules using filenames and wildcard matching.

In contrast:
- 🐍 pytask is Python-native — tasks are just regular Python functions
- ⚙️ It builds a DAG from those functions and tracks inputs/outputs automatically
- 🧱 You are declaring nodes, not scripting execution

Think of your Python files not as scripts to run, but as a way to define and wire together declarative tasks that will be executed by the pytask engine.

---

Because we defined this task in a function and loop, we can easily debug a node in the DAG by simply calling it:

In [ ]:
#| eval: false
task_download_raw_data()

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 2
      1 #| eval: false
----> 2 task_download_raw_data()

Cell In[1], line 13, in task_download_raw_data(_query)
      7 @pytask.mark.skipif(DEV_MODE, reason= "Development mode is enabled. See config.")
      8 @task(id=query.name(), name=f"Download {query.name()}")
      9 def task_download_raw_data(
     10     _query: Query = query   # The query object from the data catalog
     11 )-> Annotated[Path, data_catalog['download'][query.name()]]:
---> 13     logger = setup_logger(_query.name(), Path(f"logs/{_query.name()}.log"))
     14     output_path = BLD / f"{_query.name()}.nc"
     15     logger.info(f"Starting download for {_query.name()} to {output_path}")

File /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/src/era5_sandbox/pytask_logger.py:18, in setup_logger(name, log_file, lev

FileNotFoundError: [Errno 2] No such file or directory: '/net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/notes/logs/2024_12_nepal.log'